<a href="https://colab.research.google.com/github/hedil1/ML-Project/blob/main/Fusion_Finale_de_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import de deux data pour fusionner

import pandas as pd

df_accidents = pd.read_excel("accidents_clean_final (1).xlsx")
routes = pd.read_excel("routes_avant_mapping.xlsx")

In [ ]:
df_accidents.head()

,N°,Lieu,Lieu.Lieu,Date,Retard,Gare/PK,PK Ligne,Blessés,Tués,Ligne,Zone2,UnitéAffaure,UnitéAffaire,Sécurité,Mois
0,125994,B.Arkoub-B.B.Regba,Pleine voie,2019-01-03,28.0,2,51.200001,1.0,0.0,5,Nord Est,1,Grandes lignes,0,Janvier
1,126058,manouba-jedeida,Pleine voie,2019-01-06,19.0,2,17.900000,NaN,1.0,13,Nord,1,Grandes lignes,0,Janvier
2,126084,sidi salah-sfax,Pleine voie,2019-01-07,58.0,2,258.000000,0.0,1.0,5,Sud Est,1,Grandes lignes,0,Janvier
3,126096,bekalta-moknine,Passage à niveau,2019-01-08,65.0,4,43.720001,1.0,0.0,22,Est,1,Grandes lignes,0,Janvier
4,126097,entrée cheylus,Pleine voie,2019-01-08,45.0,2,35.299999,1.0,0.0,6,Nord Est,1,Grandes lignes,0,Janvier


In [ ]:
routes.head()

,Piste,Locales,Regionale,Nationale,Nb_intersections,Gouvernorat,Total_routes
0,0,0,0,0,0,TN,0
1,0,0,5,1,6,BR,6
2,6,1,8,1,16,MN,16
3,0,0,0,0,0,AR,0
4,3,1,1,6,11,NB,11


Les deux datasets ne sont pas directement compatibles car :
- l’un est au niveau accident
- l’autre est au niveau gouvernorat

Nous allons donc :
1. Harmoniser les gouvernorats
2. Mapper les zones vers gouvernorats
3. Agréger les données routes
4. Fusionner les datasets

In [ ]:
import pandas as pd #une bibliothèque principale en data science
import numpy as np #une bibliothèque pour les calculs numériques

In [ ]:
#lire deux data
df_accidents = pd.read_excel("accidents_clean_final (1).xlsx")
df_routes = pd.read_excel("routes_avant_mapping.xlsx")

In [ ]:
#crée un dictionnaire de regroupement géographique qui associe chaque gouvernorat tunisien (code) à sa grande zone régionale.
gov_to_zone = {
    "TN": "Nord",
    "AR": "Nord",
    "BR": "Nord",
    "MN": "Nord",
    "BZ": "Nord",
    "NB": "Nord",
    "ZG": "Nord",

    "SS": "Centre",
    "MS": "Centre",
    "MH": "Centre",
    "SF": "Centre",
    "KR": "Centre",
    "KS": "Centre",
    "SB": "Centre",

    "JN": "Ouest",
    "BJ": "Ouest",
    "KF": "Ouest",
    "SL": "Ouest",

    "GB": "Sud",
    "MD": "Sud",
    "TT": "Sud",
    "GF": "Sud",
    "TZ": "Sud",
    "KB": "Sud"
}

In [ ]:
#Création de la variable Zone
df_routes["Zone"] = df_routes["Gouvernorat"].map(gov_to_zone)

In [ ]:
#Regrouper les données par zone géographique et calculer les totaux des infrastructures.
df_routes_zone = df_routes.groupby("Zone").agg({
    "Piste": "sum",
    "Locales": "sum",
    "Regionale": "sum",
    "Nationale": "sum",
    "Nb_intersections": "sum",
    "Total_routes": "sum"
}).reset_index()

In [ ]:
#Renommage de colonne
df_accidents = df_accidents.rename(columns={"Zone2": "Zone"})

In [ ]:
#Fusion des datasets
df_final = df_accidents.merge(
    df_routes_zone,
    on="Zone",
    how="left"
)

In [ ]:
#Affiche les 5 premières lignes du dataset final
df_final.head()

,N°,Lieu,Lieu.Lieu,Date,Retard,Gare/PK,PK Ligne,Blessés,Tués,Ligne,...,UnitéAffaure,UnitéAffaire,Sécurité,Mois,Piste,Locales,Regionale,Nationale,Nb_intersections,Total_routes
0,125994,B.Arkoub-B.B.Regba,Pleine voie,2019-01-03,28.0,2,51.200001,1.0,0.0,5,...,1,Grandes lignes,0,Janvier,NaN,NaN,NaN,NaN,NaN,NaN
1,126058,manouba-jedeida,Pleine voie,2019-01-06,19.0,2,17.900000,NaN,1.0,13,...,1,Grandes lignes,0,Janvier,14.0,5.0,16.0,11.0,46.0,46.0
2,126084,sidi salah-sfax,Pleine voie,2019-01-07,58.0,2,258.000000,0.0,1.0,5,...,1,Grandes lignes,0,Janvier,NaN,NaN,NaN,NaN,NaN,NaN
3,126096,bekalta-moknine,Passage à niveau,2019-01-08,65.0,4,43.720001,1.0,0.0,22,...,1,Grandes lignes,0,Janvier,NaN,NaN,NaN,NaN,NaN,NaN
4,126097,entrée cheylus,Pleine voie,2019-01-08,45.0,2,35.299999,1.0,0.0,6,...,1,Grandes lignes,0,Janvier,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#Donne un résumé complet du Data
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   N°                97 non-null     int64         
 1   Lieu              97 non-null     object        
 2   Lieu.Lieu         97 non-null     object        
 3   Date              97 non-null     datetime64[ns]
 4   Retard            91 non-null     float64       
 5   Gare/PK           97 non-null     int64         
 6   PK Ligne          96 non-null     float64       
 7   Blessés           75 non-null     float64       
 8   Tués              77 non-null     float64       
 9   Ligne             97 non-null     object        
 10  Zone              97 non-null     object        
 11  UnitéAffaure      97 non-null     int64         
 12  UnitéAffaire      97 non-null     object        
 13  Sécurité          97 non-null     int64         
 14  Mois              97 non-nul

In [ ]:
#Affiche la liste des noms des colonnes
df_final.columns

Index(['N°', 'Lieu', 'Lieu.Lieu', 'Date', 'Retard', 'Gare/PK', 'PK Ligne',
       'Blessés', 'Tués', 'Ligne', 'Zone', 'UnitéAffaure', 'UnitéAffaire',
       'Sécurité', 'Mois', 'Piste', 'Locales', 'Regionale', 'Nationale',
       'Nb_intersections', 'Total_routes'],
      dtype='object')

In [ ]:
#Export Excel
df_final.to_excel("fusion_data_final.xlsx", index=False)

In [ ]:
#Telecherger data fusionner
from google.colab import files
files.download("fusion_data_final.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#crée un dictionnaire de correspondance (mapping) entre des noms de passages à niveau / lignes ferroviaires et leurs codes de gouvernorats ou zones.
mapping = {
"B.Arkoub-B.B.Regba": "NB",
"manouba-jedeida": "MN",
"sidi salah-sfax": "SF",
"bekalta-moknine": "MS",
"entrée cheylus": "UNK",
"El Djem-hancha": "MH",
"ezzahra mahdia": "MH",
"k.sghira-msaken": "SS",
"goulette-j.jelloud": "TN",
"Nassen-Oudna": "BR",
"B.arkoub.grombalia": "NB",
"tunis-manouba": "TN",
"B.Arada-laroussa": "SL",
"nassen-oudna": "BR",
"ghardimaou": "JN",
"sortie tébourba": "MN",
"k.sghira-enfidha": "SS",
"Arrêt S.messaoud": "UNK",
"skhira-aouinet": "SF",
"k.sghira-sousse": "SS",
"k.sghira-m.gare": "SS",
"entrée Gâafour": "SL",
"grombalia-b.cedria": "NB",
"fahs-depienne": "SL",
"PN mellaha": "UNK",
"le kef-Les salines": "KF",
"j.jelloud-b.kassa": "TN",
"Oudna-Nassen": "BR",
"nabeul-b.b.regba": "NB",
"j.jelloud-tunis": "TN",
"Khélidia": "BR",
"sortie bir el bey": "TN",
"tindja-bizerte": "BZ",
"S.abid-guergour": "UNK",
"kerker-msaken": "MS",
"Mahdia-Mahdia Z.T": "MH",
"entrée nassen": "BR",
"entrée grombalia": "NB",
"b.b.regba-nabeul": "NB",
"PN de msaken": "MS",
"Les salines-sers": "SL",
"b.kassa-cheylus": "UNK",
"b.cedria-grombalia": "NB",
"depienne-fahs": "SL",
"T.sfar-Arrêt du stad": "SF",
"b.salem-jendouba": "JN",
"monastir-s.sud": "MS",
"b.ficha-b.b.regba": "NB",
"tejrouine-jérissa": "KF",
"S.Ezzit-Sfax": "SF",
"sidi salah": "SF",
"sened-zannouch": "GF",
"hamma-chatt": "TN",
"gafsa-mg1": "GF",
"gafsa": "GF",
"entrée de Msaken": "MS",
"jend-b.salem": "JN",
"PN bardo": "TN",
"moknine-k.medyouni": "MS",
"b.salem-s.smail": "JN",
"K.sghira-M.gare": "SS",
"O.Meliz-Jendouba": "JN",
"S.Salah-sfax": "SF",
"s.sud-monastir": "MS",
"entrée medjez": "BJ",
"monastir-ksibet": "MS",
"LAROUSSA-Gâafour": "SL",
"fahs-b.arada": "SL",
"sortie ghannouch": "GB",
"j.jelloud-goulette": "TN",
"ksar helal": "MS",
"enfidha-m.gare": "SS",
"fondouk jedid": "NB",
"fahs-bou arada": "SL",
"m.gare-k.sghira": "SS",
"oudna-nassen": "BR",
"h.lif-rades": "BR",
"khéniss-ksibet": "MS",
"menzel-k sghira": "SS",
"Omar khay-hammamet": "NB",
"b ficha-enfidha": "SS",
"M.bourg-Tindja": "BZ",
"ksibet-moknine": "MS",
"pn de bejaoua": "UNK",
"PN de l'Aéroport": "TN",
"jend.ghardimaou": "JN"
}

In [ ]:
#Telecharger data
import pandas as pd

df = pd.read_excel("fusion_data_final.xlsx")

In [ ]:
#Nettoyage de la colonne Lieu
df["Lieu"] = df["Lieu"].str.lower().str.strip()

In [ ]:
#Mapping vers Gouvernorat
df["Gouvernorat"] = df["Lieu"].map(mapping)
#Gestion des valeurs manquantes
df["Gouvernorat"] = df["Gouvernorat"].fillna("UNK")

In [ ]:
#Nettoyage avancé de Lieu : convertit en string ,  met en minuscules  ,supprime espaces début/fin , remplace plusieurs espaces par un seul espace
df["Lieu"] = (
    df["Lieu"]
    .astype(str) #convertit en string ,
    .str.lower() # met en minuscules
    .str.strip() #,supprime espaces début/fin
    .str.replace(r"\s+", " ", regex=True) #remplace plusieurs espaces par un seul espace
)

In [ ]:
df.to_csv("data_finale.csv", index=False, encoding="utf-8")

In [ ]:
from google.colab import files
files.download("data_finale.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.to_excel("data_finale.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("data_finale.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
cols = ["Piste", "Locales", "Regionale", "Nationale", "Nb_intersections", "Total_routes"]

for col in cols:
    df_final[col] = df_final[col].fillna(df_final[col].median())

In [ ]:
df_final.to_excel("dataset_clean.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("dataset_clean.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>